 # Filler and Interjection Candidate Extraction Pipeline

 This notebook processes the `govnejri/kazakh_speech_mfa_punctuation` dataset to prepare a word-level cleaning pipeline for fine-tuning Microsoft VibeVoice TTS.



 ## Part 1 - Candidate Extraction & Auto-Labelling

In [ ]:
import os

# Create results directory
os.makedirs("results", exist_ok=True)

# Set Hugging Face cache location
os.environ["HF_HOME"] = "/media/storage/huggingface"
os.environ["HF_DATASETS_CACHE"] = "/media/storage/huggingface/datasets"

import json
import re
import base64
import io
import wave
import numpy as np
from datasets import load_dataset, Audio
from collections import defaultdict

# 1.1 Load dataset
print("Loading dataset...")

ds = load_dataset(
    "govnejri/kazakh_speech_mfa_punctuation",
    split="train",
    cache_dir="/media/storage/huggingface",
    verification_mode="no_checks"
)

# Fix: Prevent Hugging Face from crashing on audio decode if 'torchcodec' is missing
if "audio" in ds.features and getattr(ds.features["audio"], "decode", True):
    ds = ds.cast_column("audio", Audio(decode=False))

print(f"Loaded {len(ds)} samples.")


In [ ]:
# 1.2 Hard-coded word lists
HESITATION_FILLERS = {
    'іі', 'мм', 'уу', 'аа', 'әә', 'ээ', 'оо', 'һһ', 'ыы'
}

DISCOURSE_FILLERS = {
    'яғни', 'сол', 'ал', 'енді', 'міне', 'әрине', 'сондықтан', 'жалпы', 
    'айталық', 'типа', 'короче', 'вообще', 'просто', 'блин'
}

EXPRESSIVE_INTERJECTIONS = {
    'ой', 'әй', 'ай', 'е', 'ә', 'м', 'уһ', 'ойбай', 'әттеген-ай', 
    'алақай', 'мәссаған', 'пах-пах', 'шүкір', 'қарашы', 'тоқта', 
    'тфу', 'тфү', 'уау', 'вау', 'қап'
}

STOP_LIST = {
    'і', 'де', 'да', 'та', 'те', 'бұл', 'сол', 'ол', 'мен', 'сен', 
    'біз', 'сіз', 'екі', 'үш', 'төрт', 'бес', 'алты', 'жеті', 'сегіз', 
    'тоғыз', 'он', 'не', 'ма', 'ме', 'па', 'пе', 'ба', 'бе', 'ғой', 
    'ғана', 'жоқ', 'бар', 'кім', 'неге', 'қай', 'осы', 'және', 'деп', 
    'екен', 'емес', 'тек', 'әр', 'әлде', 'әлі', 'өте', 'тіпті', 'қазір', 
    'әрі', 'кері', 'ерте', 'кеш', 'үшін', 'дейін', 'кейін', 'бері', 
    'қарай', 'туралы', 'сияқты', 'боп', 'болып', 'кеп', 'келіп', 'ап', 
    'алып', 'ет', 'өт', 'күн', 'жер', 'ел', 'адам', 'бала', 'қол', 
    'үй', 'ат', 'от', 'су', 'тау'
}


In [ ]:
# 1.3 Helper functions
def clean_for_match(token: str) -> str:
    """Lower-case, remove leading/trailing punctuation, keep internal hyphens."""
    t = token.strip().lower()
    # Remove non-word characters at start/end (comma, period, exclamation, etc.)
    t = re.sub(r'^[\W_]+|[\W_]+$', '', t)
    return t

def is_repeated_pattern(token: str) -> bool:
    """True if token contains 3+ consecutive identical Kazakh letters."""
    return bool(re.search(r'([а-яёіңғүұқөәһ])\1{2,}', token, re.IGNORECASE))

def label_token(cleaned: str) -> str:
    """Assign one of the four tier labels based on hard-coded sets."""
    if is_repeated_pattern(cleaned) or cleaned in HESITATION_FILLERS:
        return "non_lexical_filler"
    if cleaned in DISCOURSE_FILLERS:
        return "discourse_filler"
    if cleaned in EXPRESSIVE_INTERJECTIONS:
        return "expressive_interjection"
    return "unclassified_short"


In [ ]:
# 1.4 Candidate extraction loop
print("Extracting candidates...")
candidates_full = defaultdict(list)

for idx, sample in enumerate(ds):
    words_raw = sample.get('words', [])
    if isinstance(words_raw, str):
        try:
            words = json.loads(words_raw)
        except json.JSONDecodeError:
            words = []
    else:
        words = words_raw
        
    if not words:
        continue
        
    sentence_text = sample.get("text", "")
    audio_info = sample.get("audio", {})
    audio_path = audio_info.get("path", "") if isinstance(audio_info, dict) else ""
    
    for widx, word in enumerate(words):
        original_token = word.get("text", "")
        cleaned = clean_for_match(original_token)
        
        if (len(cleaned) <= 3 or
            is_repeated_pattern(cleaned) or
            cleaned in EXPRESSIVE_INTERJECTIONS or
            cleaned in DISCOURSE_FILLERS or
            cleaned in HESITATION_FILLERS):
            
            label = label_token(cleaned)
            
            occurrence = {
                "sample_id": idx,
                "word_idx": widx,
                "start": word.get("start", 0),
                "end": word.get("end", 0),
                "original_token": original_token,
                "cleaned": cleaned,
                "sentence": sentence_text,
                "audio_path": audio_path,
                "label": label
            }
            candidates_full[cleaned].append(occurrence)

print(f"Found {sum(len(v) for v in candidates_full.values())} total candidate tokens.")
print(f"Unique candidate words: {len(candidates_full)}")

with open("results/candidates_full.json", "w", encoding="utf-8") as f:
    json.dump(candidates_full, f, ensure_ascii=False, indent=2)


In [ ]:
# 1.5 Auto-pruning of obvious function words
STOP_REMOVE = STOP_LIST - (HESITATION_FILLERS | DISCOURSE_FILLERS | EXPRESSIVE_INTERJECTIONS)

candidates_pruned = {}
auto_removed = []

for word, occurrences in candidates_full.items():
    if word in STOP_REMOVE:
        auto_removed.append(word)
    else:
        candidates_pruned[word] = occurrences

print(f"Auto-removed words: {len(auto_removed)}")
print(f"Unique candidate words after pruning: {len(candidates_pruned)}")


In [ ]:
# 1.6 Output files
with open("results/candidates_pruned.json", "w", encoding="utf-8") as f:
    json.dump(candidates_pruned, f, ensure_ascii=False, indent=2)

with open("results/auto_removed_words.txt", "w", encoding="utf-8") as f:
    for word in sorted(auto_removed):
        f.write(f"{word}\n")

categories = {
    "non_lexical_filler": [],
    "discourse_filler": [],
    "expressive_interjection": [],
    "unclassified_short": []
}

word_counts = []

for word, occurrences in candidates_pruned.items():
    label = occurrences[0]["label"]
    categories[label].append(word)
    word_counts.append((word, label, len(occurrences)))

for cat_name, words in categories.items():
    with open(f"results/{cat_name}s.txt", "w", encoding="utf-8") as f:
        cat_words_sorted = sorted([(w, len(candidates_pruned[w])) for w in words], key=lambda x: x[1], reverse=True)
        for w, count in cat_words_sorted:
            f.write(f"{w}\n")

word_counts.sort(key=lambda x: x[2], reverse=True)
with open("results/words_for_manual_review.tsv", "w", encoding="utf-8") as f:
    f.write("word\tlabel\tcount\n")
    for word, label, count in word_counts:
        f.write(f"{word}\t{label}\t{count}\n")

print("\n--- Summary ---")
total_pruned_tokens = sum(len(v) for v in candidates_pruned.values())
print(f"Total candidate tokens (after pruning): {total_pruned_tokens}")
print(f"Unique words before pruning: {len(candidates_full)}")
print(f"Unique words after pruning: {len(candidates_pruned)}")
print("\nCounts per category (unique words):")
for cat_name, words in categories.items():
    print(f"  {cat_name}: {len(words)}")

print("\nTop 20 words by frequency:")
for word, label, count in word_counts[:20]:
    print(f"  {word} ({label}): {count}")


 ## Part 2 - LLM Prompt Generation

In [ ]:
llm_jsonl_path = "results/llm_prompts.jsonl"
llm_txt_path = "results/llm_prompts.txt"

unclassified_words = categories["unclassified_short"]
unclassified_words_sorted = sorted([(w, len(candidates_pruned[w])) for w in unclassified_words], key=lambda x: x[1], reverse=True)

with open(llm_jsonl_path, "w", encoding="utf-8") as f_jsonl, \
     open(llm_txt_path, "w", encoding="utf-8") as f_txt:
    
    for word, _ in unclassified_words_sorted:
        occurrences = candidates_pruned[word]
        examples = []
        seen_sentences = set()
        
        for occ in occurrences:
            sent = occ["sentence"].strip()
            if sent and sent not in seen_sentences:
                examples.append(sent)
                seen_sentences.add(sent)
            if len(examples) >= 3:
                break
                
        jsonl_obj = {"word": word, "examples": examples}
        f_jsonl.write(json.dumps(jsonl_obj, ensure_ascii=False) + "\n")
        
        f_txt.write(f"Word: {word}\nSentences:\n")
        for i, ex in enumerate(examples, 1):
            f_txt.write(f"  {i}. {ex}\n")
        f_txt.write("\n")

print(f"LLM prompts generated for {len(unclassified_words)} unclassified words.")


 ## Part 3 - Interactive In-Notebook Review (Recommended)

 The following cells use `pandas` and `ipywidgets` to create an interactive review tool.

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display, Audio
import wave
import io
import os
import json

# Load the TSV review file
df = pd.read_csv("results/words_for_manual_review.tsv", sep="\t")
print(f"Loaded {len(df)} words for review.")

# Load the candidates JSON so this cell can run independently
with open("results/candidates_pruned.json", "r", encoding="utf-8") as f:
    candidates_pruned = json.load(f)



In [ ]:
def get_audio_snippet(audio_path, start_sec, end_sec):
    """Return IPython Audio widget for a given audio file and time slice."""
    try:
        if not audio_path or not os.path.exists(audio_path):
            return "No audio path available"
        
        with wave.open(audio_path, 'rb') as w:
            n_channels = w.getnchannels()
            sampwidth = w.getsampwidth()
            framerate = w.getframerate()
            
            start_frame = int(start_sec * framerate)
            end_frame = int(end_sec * framerate)
            n_frames = max(1, end_frame - start_frame)
            
            w.setpos(start_frame)
            audio_frames = w.readframes(n_frames)
            
            out_io = io.BytesIO()
            with wave.open(out_io, 'wb') as out_w:
                out_w.setnchannels(n_channels)
                out_w.setsampwidth(sampwidth)
                out_w.setframerate(framerate)
                out_w.writeframes(audio_frames)
            
            return Audio(data=out_io.getvalue(), autoplay=False)
    except Exception as e:
        return f"Error loading audio: {e}"

# Dropdown to select word
word_dropdown = widgets.Dropdown(
    options=sorted(df['word'].dropna().unique()),
    description='Word:',
)

# Output area
audio_output = widgets.Output()

def on_word_change(change):
    word = change['new']
    with audio_output:
        clear_output(wait=True)
        # Find occurrences of this word from candidates_pruned
        occs = candidates_pruned.get(word, [])
        if not occs:
            print("No occurrences found.")
            return
        # Show first 3 occurrences
        for i, occ in enumerate(occs[:3]):
            print(f"--- Occurrence {i+1} ---")
            print(f"Sentence: {occ['sentence']}")
            snippet = get_audio_snippet(occ['audio_path'], occ['start'], occ['end'])
            if isinstance(snippet, str):
                print(snippet)
            else:
                display(snippet)

word_dropdown.observe(on_word_change, names='value')
print("Select a word from the dropdown to review audio snippets:")
display(word_dropdown, audio_output)

# Trigger initial display
if word_dropdown.options:
    word_dropdown.value = word_dropdown.options[0]


In [ ]:
# Find specific word occurrences (e.g., "ааз")
target_word = "ааз"
print(f"Occurrences for '{target_word}':")
if target_word in candidates_pruned:
    occs = candidates_pruned[target_word]
    print(f"Found {len(occs)} occurrences.")
    for i, occ in enumerate(occs):
        print(f"\n--- Occurrence {i+1} ---")
        print(f"Sample ID (index): {occ['sample_id']}")
        print(f"Sentence: {occ['sentence']}")
        print(f"Audio Path: {occ['audio_path']}")
        print(f"Time: {occ['start']}s - {occ['end']}s")
        snippet = get_audio_snippet(occ['audio_path'], occ['start'], occ['end'])
        if isinstance(snippet, str):
            print(snippet)
        else:
            display(snippet)
        
        # Limit to first 5 to avoid spamming the notebook
        if i >= 4:
            print("\n... (showing first 5 occurrences only)")
            break
else:
    print(f"'{target_word}' not found in candidates_pruned.")
